# Finite-level Fermi–Dirac occupations in a one-dimensional box

**Prerequisite:** [Finite-level occupations and spatial density](../../../docs/lecture-notes/qm/piab1d/piab1d__fermions.md)

This maintained laboratory extracts the occupation and spatial-density analysis from the historical PIAB1D visualization notebooks. It uses an **illustrative independent-particle model**: 100 spinless single-particle box levels, a target expected occupation of 8, and no interactions or implicit degeneracy.

## Learning objectives

1. Solve a finite-level Fermi–Dirac particle-number equation.
2. Inspect occupation as a function of discrete level energy.
3. Construct the occupation-weighted spatial number density.
4. Verify that the integrated density equals the target expected occupation.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.qm.piab1d.base import Piab1D
from projectkoios.physkit.qm.piab1d.tise_analytical.solution import (
    Piab1DAnalyticalSolution,
)
from projectkoios.physkit.thermal.statmech.qm import (
    FermiDiracChemicalPotentialSolver,
    FermiDiracState,
    OccupiedEigenfunctionDensityEvaluator,
)
from projectkoios.physkit.units import (
    MODEL_SYSTEM_UNIT_CONVERTER,
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    UnitSystem,
    VectorQuantity,
)

In [ ]:
temperature_kelvin = 2000.0
target_occupation = ScalarQuantity(magnitude=8.0, unit=Unitless())
level_count = 100

model = Piab1D(
    length=1.0e-9,
    mass=SI.me0,
    unit_system=UnitSystem.SI,
)
stationary_states = Piab1DAnalyticalSolution(model=model).evaluate(count=level_count)
thermal_energy = ScalarQuantity(
    magnitude=SI.k_B * temperature_kelvin, unit=model.unit_system.energy_unit
)
occupation_result = FermiDiracChemicalPotentialSolver().solve(
    energies=stationary_states.energies,
    thermal_energy=thermal_energy,
    target_occupation=target_occupation,
)

assert abs(occupation_result.occupation_residual.magnitude) <= 1.0e-12
assert np.isclose(
    np.sum(occupation_result.occupations.magnitude),
    target_occupation.magnitude,
    rtol=0.0,
    atol=1.0e-12,
)

In [ ]:
fermi_dirac_state = FermiDiracState(
    thermal_energy=thermal_energy,
    chemical_potential=occupation_result.chemical_potential,
)
dense_energy_values = np.linspace(
    0.0,
    1.15 * occupation_result.chemical_potential.magnitude,
    1000,
    dtype=np.float64,
)
dense_energies = VectorQuantity(
    magnitude=dense_energy_values,
    unit=model.unit_system.energy_unit,
)
dense_occupations = fermi_dirac_state.evaluate_occupations(
    energies=dense_energies,
)

In [ ]:
coordinate_values_metres = np.linspace(0.0, model.length, 1001, dtype=np.float64)
coordinates = VectorQuantity(
    magnitude=coordinate_values_metres, unit=model.unit_system.length_unit
)
eigenfunctions = stationary_states.eigenfunctions(coordinates=coordinates)
density_evaluation = OccupiedEigenfunctionDensityEvaluator().evaluate(
    eigenfunctions=eigenfunctions,
    occupations=occupation_result.occupations,
)
number_density = density_evaluation.density
represented_particle_number = float(
    np.trapezoid(number_density.magnitude, x=coordinate_values_metres)
)
position_probability_density = number_density.magnitude / target_occupation.magnitude

assert np.isclose(
    represented_particle_number,
    target_occupation.magnitude,
    rtol=0.0,
    atol=2.0e-12,
)
assert np.isclose(
    np.trapezoid(position_probability_density, x=coordinate_values_metres),
    1.0,
    rtol=0.0,
    atol=2.0e-13,
)
represented_particle_number

In [ ]:
energies_electron_volts = MODEL_SYSTEM_UNIT_CONVERTER.convert_vector(
    stationary_states.energies,
    PhysicalUnit(expression="electron_volt"),
)
chemical_potential_electron_volts = MODEL_SYSTEM_UNIT_CONVERTER.convert_scalar(
    occupation_result.chemical_potential,
    PhysicalUnit(expression="electron_volt"),
)
dense_energies_electron_volts = MODEL_SYSTEM_UNIT_CONVERTER.convert_vector(
    dense_energies,
    PhysicalUnit(expression="electron_volt"),
)

figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.75))

axes[0].plot(
    dense_energies_electron_volts.magnitude,
    dense_occupations.occupations.magnitude,
    label="continuous Fermi--Dirac function",
)
axes[0].scatter(
    energies_electron_volts.magnitude,
    occupation_result.occupations.magnitude,
    s=20,
    label="discrete box levels",
    zorder=3,
)
axes[0].axvline(
    chemical_potential_electron_volts.magnitude,
    color="black",
    linestyle="--",
    label=r"chemical potential $\mu$",
)
axes[0].set(
    xlim=(0.0, 1.15 * chemical_potential_electron_volts.magnitude),
    xlabel="single-particle energy (eV)",
    ylabel="occupation",
    title="Fermi--Dirac occupations",
)
axes[0].grid(True)
axes[0].legend(fontsize="small")

axes[1].plot(
    coordinate_values_metres * 1.0e9,
    number_density.magnitude * 1.0e-9,
)
axes[1].set(
    xlabel="position (nm)",
    ylabel=r"number density (nm$^{-1}$)",
    title="Occupation-weighted spatial density",
)
axes[1].grid(True)

figure.tight_layout()
plt.show()

## Interpretation

The first eight spinless box levels are nearly fully occupied for the selected scale and temperature, while higher levels are nearly empty. The continuous Fermi--Dirac curve exposes how the discrete box levels sample the same occupation function. Their squared standing-wave orbitals combine into an eight-peak spatial number density. Integrating that density returns the target expected occupation rather than one; division by the target produces a normalized position probability density for a particle sampled from this modeled population.

This is a finite independent-particle occupation analysis. It does not introduce electron–electron interactions, self-consistency, spin degeneracy, or a continuum density of states.

## Exercises

1. Increase the temperature and describe the broadening of the occupation transition.
2. Compare targets of 1, 4, and 8 while keeping the represented level inventory fixed.
3. Apply an explicit degeneracy convention and adjust the target equation consistently.
4. Compare Fermi–Dirac occupations with `CanonicalLevelProbabilityModel` and explain why the normalized quantities answer different questions.
